# Platypus: would a pretrained base lift the reader?

The self-trained reader (usefulness pseudo-labels, no relation labels) on the internal states of small PRETRAINED language models, against our best label-free reader (word counts: AUC about 0.65) and each model's ceiling with true labels (evaluation only). About 20-40 minutes on an A100.

**Secrets:** nothing here needs a key; the models are public. Never paste a key into this notebook.

**Caveat:** these models were pretrained on the web, Wikipedia included, so part of a gain can be knowing the fact rather than reading it. The pairs-seen-once rows help; a clean test of reading needs invented names (a later step).

Copyright Christian Bird. Apache 2.0.

In [ ]:
REPO = 'https://github.com/chrisbirduk/platypus.git'
BRANCH = 'cg27a-distill'
USE_DRIVE = True
DRIVE_DIR = '/content/drive/MyDrive/platypus'
MODELS = 'gpt2,HuggingFaceTB/SmolLM2-360M,Qwen/Qwen2.5-0.5B,Qwen/Qwen2.5-1.5B'   # comma list; add Qwen/Qwen2.5-3B on an A100 if time allows
DOCS = 20000           # distant documents read (records come from these; the last 20% held out)

In [ ]:
import subprocess, torch, os
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive'); os.makedirs(DRIVE_DIR, exist_ok=True)
if not os.path.isdir('/content/platypus/.git'):
    !git clone -q --branch {BRANCH} {REPO} /content/platypus
else:
    !git -C /content/platypus fetch -q origin {BRANCH} && git -C /content/platypus checkout -q {BRANCH} && git -C /content/platypus reset -q --hard origin/{BRANCH}
%cd /content/platypus
!git log -1 --oneline
!pip -q install transformers scikit-learn

## The distant set (DocRED's distant training documents, 91 MB, checked against its SHA-256)

In [ ]:
import hashlib, shutil, urllib.request
DIST_URL = 'https://media.githubusercontent.com/media/multiful/Information_Extraction/main/docred_data/data/train_distant.json.gz'
DIST_SHA = 'c420c0429310583cfc9459f7daa26b1f4c11ff5c7a1481aa64ab9db2b296b905'
os.makedirs('/content/distant', exist_ok=True); dst = '/content/distant/train_distant.json.gz'
cache = f'{DRIVE_DIR}/train_distant.json.gz' if USE_DRIVE else dst
sha = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()
if not (os.path.exists(cache) and sha(cache) == DIST_SHA):
    urllib.request.urlretrieve(DIST_URL, cache)
assert sha(cache) == DIST_SHA, 'distant set: checksum mismatch'
if cache != dst: shutil.copy(cache, dst)
print('distant set ready:', dst, os.path.getsize(dst) // 2**20, 'MB')


## Run

In [ ]:
os.environ.update(PR_DOCS=str(DOCS), PR_MODELS=MODELS)
!python tools/pretrained_reader.py /content/distant/train_distant.json.gz 2>&1 | grep -v 'Loading weights' | tee /content/pretrained_reader.log
import shutil
if USE_DRIVE: shutil.copy('/content/pretrained_reader.log', f'{DRIVE_DIR}/pretrained_reader.log')